In [1]:
#run spark.stop before running this notebook. some of the configs won't take if an active spark sessions is grabbed or running. 
#spark.stop()

In [2]:
import pyspark
import os
import pandas as pd
from torchvision.io import read_image
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import datasets, models
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import pyarrow


from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import FloatType

import math
from statistics import NormalDist

In [3]:
class CNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = torch.nn.Sequential(
            #Input = 3 x 32 x 32, Output = 32 x 32 x 32 
            #new input 3x600x375 output=
            torch.nn.Conv2d(in_channels = 3, out_channels = 6, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 32 x 32 x 32, Output = 32 x 16 x 16 
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 32 x 16 x 16, Output = 64 x 16 x 16 
            torch.nn.Conv2d(in_channels = 6, out_channels = 12, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 16 x 16, Output = 64 x 8 x 8 
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 64 x 8 x 8, Output = 64 x 8 x 8 
            torch.nn.Conv2d(in_channels = 12, out_channels = 24, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 8 x 8, Output = 64 x 4 x 4 
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 64 x 8 x 8, Output = 64 x 8 x 8 
            torch.nn.Conv2d(in_channels = 24, out_channels = 48, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 8 x 8, Output = 64 x 4 x 4 
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 64 x 8 x 8, Output = 64 x 8 x 8 
            torch.nn.Conv2d(in_channels = 48, out_channels = 96, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 8 x 8, Output = 64 x 4 x 4 
            torch.nn.MaxPool2d(kernel_size=2),

            torch.nn.Flatten(),

            torch.nn.Linear(96*11*18, 128),
            torch.nn.ReLU(),
            torch.nn.BatchNorm1d(128),




            #torch.nn.Linear(128, 128),
            #torch.nn.ReLU(),
            #torch.nn.BatchNorm1d(128),



            torch.nn.Linear(128, 2),
            torch.nn.ReLU(),
            #torch.nn.BatchNorm1d(2),
            #torch.nn.Dropout(0.2),


            torch.nn.Softmax(dim=1)

        )

    def forward(self, x):

            

        return self.model(x)

In [4]:
#tuning knob 1. put a very between 1 and 10, depending on sys cores. this also reparitions the df for sql queries
threads = 2

In [5]:
spark = pyspark.sql.SparkSession.builder.master(f"local[{threads}]").appName("silver").config("spark.executor.memory", "10g").config("spark.driver.memory", "10g").getOrCreate()

print(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/09/15 15:12:54 WARN Utils: Your hostname, research-ub22, resolves to a loopback address: 127.0.1.1; using 192.168.0.54 instead (on interface ens18)
25/09/15 15:12:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/15 15:12:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
model = CNN().to(device)
model.load_state_dict(torch.load('../mymodel', map_location=torch.device(device)))
model.eval() # Set the model to evaluation mode

bc_model = spark.sparkContext.broadcast(model)

cpu


In [7]:
#second_model
res18_model = models.resnet18(pretrained=True).to(device)
res18_model.fc = nn.Linear(res18_model.fc.in_features, 2 )
res18_model.load_state_dict(torch.load('../resnet18_model.pth', map_location=torch.device(device)))
res18_model.eval() # Set the model to evaluation mode

bc_res18_model = spark.sparkContext.broadcast(res18_model)

/home/starfuryz2040/DataspellProjects/research/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/starfuryz2040/DataspellProjects/research/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [8]:
print(device)

cpu


NORMAL UDF (doesnt work in spark sql)

In [9]:
def predict_tensor(tensor_data):
    tensor = torch.Tensor(tensor_data)
    
    #channel, h, w = tensor.shape
  
    with torch.no_grad():  # Disable gradient calculation for inference
        prediction = model(tensor.unsqueeze(0).to(device))
        _, y_pred = torch.max(prediction.data, 1)
        
    return float(y_pred)
    

    
    

predict_udf = udf(predict_tensor, FloatType())

In [10]:
spark.udf.register("predict_udf", predict_udf)

In [11]:
def predict_res18_tensor(tensor_data):
    tensor = torch.Tensor(tensor_data)
    
    #channel, h, w = tensor.shape
  
    with torch.no_grad():  # Disable gradient calculation for inference
        prediction = res18_model(tensor.unsqueeze(0).to(device))
        _, y_pred = torch.max(prediction.data, 1)
        
    return float(y_pred)
    

    
    

predict_udf = udf(predict_tensor, FloatType())

In [12]:
spark.udf.register("predict_res18_udf", predict_udf)

VECTORIZED UDF (WORKS IN SPARK SQL)

In [13]:
@pandas_udf(returnType=FloatType())
def v_predict_tensor(tensor_data):
    

    
    input_tensor = torch.Tensor(tensor_data.tolist())

  

    
    with torch.no_grad():  # Disable gradient calculation for inference
    
        predictions = model(input_tensor.to(device))
        _, y_pred = torch.max(predictions.data, 1)
    
    #return y_pred
    return pd.Series(y_pred.squeeze().tolist())


In [14]:
spark.udf.register("v_predict_udf", v_predict_tensor)

In [15]:
@pandas_udf(returnType=FloatType())
def v_predict_res18_tensor(tensor_data):
    

    
    input_tensor = torch.Tensor(tensor_data.tolist())

  

    
    with torch.no_grad():  # Disable gradient calculation for inference
    
        predictions = model(input_tensor.to(device))
        _, y_pred = torch.max(predictions.data, 1)
    
    #return y_pred
    return pd.Series(y_pred.squeeze().tolist())

In [16]:
spark.udf.register("v_predict_res18_udf", v_predict_res18_tensor)

In [17]:
#tuning knob 2. put a .limit(20) at the end of the data set loads to reduce memory footprint for dev stuff 
#all_images_labeled = spark.read.parquet("./silver/all_images_labeled.parquet")
labeled_training_set = spark.read.parquet("./silver/labeled_training_set.parquet")
labeled_training_set.cache()
labeled_training_set.count()

998

examples of dataframe api udf calls. Faster than sql, but will need to do more dataframe manipulation thats not as easy as spark.sql

In [18]:
test = labeled_training_set.withColumn("mylabel", predict_udf(labeled_training_set['img_tensor']))
test.show()

+-------------------+--------------------+----------------+-------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+-------+
|               name|          img_tensor|prelabeled_fraud|                 id|    person_name|             address|  birthday|gender|ethnicity|class|issue_date|expire_date|eye_color|hair_color|is_donor|is_veteran|license_number|height_inches|weight_lbs|mylabel|
+-------------------+--------------------+----------------+-------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+-------+
|generated_fake_2_49|[[[0.98039216, 0....|               1|generated_fake_2_49| ABIGAIL MILLER|38 Silver Creek R...| 12/3/2006|     F|     NULL|ERROR|11/13/2021| 11/13/2026|      BRO|      NULL|   FALSE|     FAL

In [19]:
test.schema

StructType([StructField('name', StringType(), True), StructField('img_tensor', ArrayType(ArrayType(ArrayType(FloatType(), True), True), True), True), StructField('prelabeled_fraud', IntegerType(), True), StructField('id', StringType(), True), StructField('person_name', StringType(), True), StructField('address', StringType(), True), StructField('birthday', StringType(), True), StructField('gender', StringType(), True), StructField('ethnicity', StringType(), True), StructField('class', StringType(), True), StructField('issue_date', StringType(), True), StructField('expire_date', StringType(), True), StructField('eye_color', StringType(), True), StructField('hair_color', StringType(), True), StructField('is_donor', StringType(), True), StructField('is_veteran', StringType(), True), StructField('license_number', StringType(), True), StructField('height_inches', IntegerType(), True), StructField('weight_lbs', StringType(), True), StructField('mylabel', FloatType(), True)])

In [20]:
test2 = labeled_training_set.withColumn("mylabel", v_predict_tensor(labeled_training_set['img_tensor']))
test2.show()

/tmp/ipykernel_2973/1397212364.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)


+-------------------+--------------------+----------------+-------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+-------+
|               name|          img_tensor|prelabeled_fraud|                 id|    person_name|             address|  birthday|gender|ethnicity|class|issue_date|expire_date|eye_color|hair_color|is_donor|is_veteran|license_number|height_inches|weight_lbs|mylabel|
+-------------------+--------------------+----------------+-------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-------------+----------+-------+
|generated_fake_2_49|[[[0.98039216, 0....|               1|generated_fake_2_49| ABIGAIL MILLER|38 Silver Creek R...| 12/3/2006|     F|     NULL|ERROR|11/13/2021| 11/13/2026|      BRO|      NULL|   FALSE|     FAL

In [21]:
#last tuning know, change reparition based on your system specs and the size of the data being read in. spark will usually do around 20 by default. may or may not work well for you. higher will protect against OOM errors. Lower will make your processing time alot higher
labeled_training_set = labeled_training_set.repartition(threads*10)
labeled_training_set.cache()
labeled_training_set.count()

998

In [22]:
#labeled_training_set.rdd.getNumPartitions()

examples of using udf in sql queries

In [23]:
spark.sql("select gender, predict_udf(img_tensor) as my_predictions from {labeled_training_set}", labeled_training_set=labeled_training_set).show()

+------+--------------+
|gender|my_predictions|
+------+--------------+
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     M|           1.0|
|     M|           1.0|
|     M|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     M|           0.0|
|     F|           0.0|
+------+--------------+
only showing top 20 rows


In [24]:
spark.sql("select gender, v_predict_udf(img_tensor) as my_predictions from {labeled_training_set}", labeled_training_set=labeled_training_set).show()

/tmp/ipykernel_2973/1397212364.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)


+------+--------------+
|gender|my_predictions|
+------+--------------+
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     M|           1.0|
|     M|           1.0|
|     M|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     M|           0.0|
|     F|           0.0|
+------+--------------+
only showing top 20 rows


In [25]:
spark.sql("with cte as (select gender, predict_udf(img_tensor) as my_predictions from {labeled_training_set}) select gender, my_predictions, count(my_predictions) as count_predictions from cte group by gender, my_predictions", labeled_training_set=labeled_training_set).show()

+------+--------------+-----------------+
|gender|my_predictions|count_predictions|
+------+--------------+-----------------+
|     F|           1.0|              104|
|     F|           0.0|              442|
|     M|           1.0|               92|
|     M|           0.0|              357|
| ERROR|           1.0|                3|
+------+--------------+-----------------+



In [26]:
spark.sql("with cte as (select gender, v_predict_udf(img_tensor) as my_predictions from {labeled_training_set}) select gender, my_predictions, count(my_predictions) as count_predictions from cte group by gender, my_predictions", labeled_training_set=labeled_training_set).show()


/tmp/ipykernel_2973/1397212364.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
/tmp/ipykernel_2973/1397212364.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)


+------+--------------+-----------------+
|gender|my_predictions|count_predictions|
+------+--------------+-----------------+
|     F|           1.0|              104|
|     F|           0.0|              442|
|     M|           1.0|               92|
|     M|           0.0|              357|
| ERROR|           1.0|                3|
+------+--------------+-----------------+



In [27]:
#using res18
spark.sql("select gender, v_predict_res18_udf(img_tensor) as my_predictions from {labeled_training_set}", labeled_training_set=labeled_training_set).show()

+------+--------------+
|gender|my_predictions|
+------+--------------+
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     M|           1.0|
|     M|           1.0|
|     M|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     F|           0.0|
|     F|           0.0|
|     M|           0.0|
|     M|           0.0|
|     F|           0.0|
+------+--------------+
only showing top 20 rows


In [28]:
spark.sql("with cte as (select gender, v_predict_res18_udf(img_tensor) as my_predictions from {labeled_training_set}) select gender, my_predictions, count(my_predictions) as count_predictions from cte group by gender, my_predictions", labeled_training_set=labeled_training_set).show()

/tmp/ipykernel_2973/2464696233.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)


+------+--------------+-----------------+
|gender|my_predictions|count_predictions|
+------+--------------+-----------------+
|     F|           1.0|              104|
|     F|           0.0|              442|
|     M|           1.0|               92|
|     M|           0.0|              357|
| ERROR|           1.0|                3|
+------+--------------+-----------------+



In [29]:
def pdf_udf(gender, weight_lbs):
    
    #def normpdf(x, mean, sd):
    #    var = float(sd)**2
    #    denom = (2*math.pi*var)**.5
    #    num = math.exp(-(float(x)-float(mean))**2/(2*var))
    #    return num/denom
    
    #randomly taken from google, do not claim this to be sci accurate on stats
    male_mean = 200
    male_dev = 40
    female_mean = 170
    female_dev = 51

    gender = gender.upper()
    
    if gender == 'M':
        a = NormalDist(male_mean, male_dev).cdf(int(weight_lbs)+5)
        b = NormalDist(male_mean, male_dev).cdf(int(weight_lbs)-5)
        probability_density = abs(a-b)
        
    else: 
        a = NormalDist(female_mean, female_dev).cdf(int(weight_lbs)+5)
        b = NormalDist(female_mean, female_dev).cdf(int(weight_lbs)-5)
        probability_density = abs(a-b)
    
    return round(probability_density, 3)

In [30]:
spark.udf.register("pdf_udf_sql", pdf_udf)

<function __main__.pdf_udf(gender, weight_lbs)>

In [31]:
spark.sql('select gender, weight_lbs, pdf_udf_sql(gender, weight_lbs) from {labeled_training_set}', labeled_training_set=labeled_training_set).show()

+------+----------+-------------------------------+
|gender|weight_lbs|pdf_udf_sql(gender, weight_lbs)|
+------+----------+-------------------------------+
|     M|       124|                          0.017|
|     M|       163|                          0.065|
|     F|       128|                          0.056|
|     F|       179|                          0.077|
|     F|       175|                          0.078|
|     M|       128|                           0.02|
|     M|       103|                          0.005|
|     M|       138|                           0.03|
|     M|       155|                          0.053|
|     M|       162|                          0.063|
|     M|       229|                          0.077|
|     M|       186|                          0.094|
|     M|       211|                          0.096|
|     M|       142|                          0.035|
|     F|       140|                          0.066|
|     M|       100|                          0.004|
|     F|    

In [32]:
spark.sql("with cte1 as (select gender, weight_lbs, img_tensor, pdf_udf_sql(gender, weight_lbs) as w_prob from {labeled_training_set}), cte2 as (select *, v_predict_udf(img_tensor) as fraud from cte1) select gender, fraud, avg(w_prob), max(w_prob), min(w_prob) from cte2 group by gender, fraud" , labeled_training_set=labeled_training_set).show()

/tmp/ipykernel_2973/1397212364.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
/tmp/ipykernel_2973/1397212364.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)


+------+-----+-------------------+-----------+-----------+
|gender|fraud|        avg(w_prob)|max(w_prob)|min(w_prob)|
+------+-----+-------------------+-----------+-----------+
| ERROR|  1.0|0.06133333333333333|      0.077|      0.034|
|     F|  0.0|0.06374886877828054|      0.078|      0.031|
|     F|  1.0|           0.063875|      0.078|        0.0|
|     M|  0.0|0.05880672268907563|      0.099|      0.004|
|     M|  1.0|0.06670652173913044|      0.099|        0.0|
+------+-----+-------------------+-----------+-----------+

